In [1]:
import json
import pandas as pd
import copy
import re

## Do not mention the file extension

In [2]:
filename1 = '../prodigy/labelled_data/unseen_queries/unseen_queries_3_8_23_a'

In [4]:
with open(filename1 + '.jsonl', 'r') as fp:
    file1 = [json.loads(line) for line in fp]

In [5]:
def remove_substring(input_string, remove_substring):
    input_list = input_string.split()
    remove_sublist = remove_substring.split()
        
    # Find the starting index of the sublist to remove
    start_index = -1
    for i in range(len(input_list) - len(remove_sublist) + 1):
        if input_list[i:i+len(remove_sublist)] == remove_sublist:
            start_index = i
            break
            
    if start_index != -1:
        # Remove the sublist from the output list
        del input_list[start_index:start_index + len(remove_sublist)]
        
    output_text = " ".join(input_list)
    return output_text

In [6]:
def get_labelled_df(prodigy_labelled_data):
    prodigy_labels = copy.deepcopy(prodigy_labelled_data)
    
    labelled_data = []
    for i in prodigy_labels:
        
        row = {
            "QUERY": "",
            "BRAND": "",
            "POLYMER": "",
            "PROPERTY": "",
            "FEATURE": "",
            "FILLER": "",
            "GRADE": "",
            "CERTIFICATION": "",
            "COMPETITOR_GRADE": "",
            "APPLICATION": "",
            "MODIFIER": "",
            "FILLER_PERCENTAGE": "",
            "UNIDENTIFIED": "",
            "spacy_format": "",
            "accept": True
          }
            
        row['QUERY'] += i['text']
        row['UNIDENTIFIED'] += i['text']
        
        if i["answer"] == "accept":
            entities = {'entities': []}
            to_remove_texts = []
            for j in i['spans']:
                entities['entities'].append((j['start'], j['end'], j['label']))

                span_text = i['text'][j['start']:j['end']]
                if row[j['label']]:
                    row[j['label']] += "|" + span_text
                else:
                    row[j['label']] += span_text

    #             pattern = r"\b{}\b".format(re.escape(span_text))
    #             row['UNIDENTIFIED'] = re.sub(pattern, '', row['UNIDENTIFIED'])
                to_remove_texts.append(span_text)
#                 to_remove_texts += [span_text]
#                 to_remove_texts += [span_text+i for i in [".", ",", ":", "/", "?", ";", '"', ".,", ",."]]
#                 to_remove_texts += [i+span_text for i in [",", ":", "/", ";", '"', ".,", ",."]]
            
            to_remove_texts = sorted(to_remove_texts, key=len, reverse=True)
            for r in to_remove_texts:
#                 row['UNIDENTIFIED'] = remove_substring(row['UNIDENTIFIED'], r)
                row['UNIDENTIFIED'] = row['UNIDENTIFIED'].replace(r, ' ', 1)
            
            row['spacy_format'] = (i['text'], entities)
            row['UNIDENTIFIED'] = re.sub("(\s+)", " ", row['UNIDENTIFIED'].strip())
        else:
            row['accept']= False
            
        labelled_data.append(row)

    labelled_df = pd.DataFrame(labelled_data)
    return labelled_df

In [9]:
df1 = get_labelled_df(file1)

In [10]:
cols = ['QUERY', 'BRAND', 'POLYMER', 'PROPERTY', 'FEATURE', 'FILLER', 'GRADE',
       'CERTIFICATION', 'COMPETITOR_GRADE', 'APPLICATION', 'MODIFIER',
       'FILLER_PERCENTAGE', 'UNIDENTIFIED', 'spacy_format']

In [11]:
df1[cols] = df1[cols].astype(str).applymap(lambda x: '"' + x + '"')

In [12]:
df1.to_excel(filename1 + ".xlsx", index=False)